In [1]:
import requests
from bs4 import BeautifulSoup #za pretrazivanje i čitanje HTML sadrzaa
import pandas as pd

In [4]:
url = "https://itbase.ba/poslovi"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

In [5]:
response = requests.get(url, headers=headers)
print(f"Status kod odgovara: {response.status_code}")

Status kod odgovara: 200


In [6]:
soup = BeautifulSoup(response.text, "html.parser")

In [7]:
print("Naslov stranice: ", soup.title.string)

Naslov stranice:  IT poslovi - Pronađi visoko plaćeni posao  | IT Base


In [17]:
podatci_oglasi = []
svi_linkovi = soup.find_all("a", href=True)

for a in svi_linkovi:
    href = a['href']
    if "/posao" in href or "/poslovi" in href:
        naslov_tekst = a.get_text(separator=" ", strip=True)

        if len(naslov_tekst) > 3 and naslov_tekst != "POSLOVI":
            full_url = href if href.startswith("http") else "https://www.itbase.ba" + href

            podatci_oglasi.append({
                "naslov_i_opis": naslov_tekst,
                "link": full_url
            })

if not podatci_oglasi:
    svi_divovi = soup.find_all("div")
    for d in  svi_divovi:
        tekst = d.get_text(separator=" | ", strip=True)
        if any(grad in tekst for grad in ["MOSTAR", "SARAJEVO", "REMOTE", "BANJA LUKA", "ZAGREB"]):
            if len(tekst) < 300 and len(tekst) > 20: 
                podaci_oglasi.append({
                    "naslov_i_opis": tekst,
                    "link": url
                })

df = pd.DataFrame(podatci_oglasi).drop_duplicates(subset=["naslov_i_opis"])
print(f"Uspješno pronađeno {len(df)} oglasa!\n")
df.head(10)

Uspješno pronađeno 29 oglasa!



,naslov_i_opis,link
0,Poslovi,https://itbase.ba/poslovi
1,Objavi oglas,https://itbase.ba/poslovi/objava
4,Remote BDR Admin,https://itbase.ba/poslovi/r/remote-bdr-admin/2156
5,Frontend (React) Developer,https://itbase.ba/poslovi/r/frontend-react-dev...
6,Senior Solution Engineer,https://itbase.ba/poslovi/r/senior-solution-en...
8,Remote HR Administrator,https://itbase.ba/poslovi/r/remote-hr-administ...
9,Remote Implementation Consultant (Part Time),https://itbase.ba/poslovi/r/remote-implementat...
10,Remote Intermediate Python Developer,https://itbase.ba/poslovi/r/remote-intermediat...
11,Remote Intermediate Ruby Developer,https://itbase.ba/poslovi/r/remote-intermediat...
12,Remote Senior Front-End JavaScript Developer,https://itbase.ba/poslovi/r/remote-senior-fron...


In [18]:
import re

In [24]:
tehnologije_lista = ["Python", "SQL", "React", "JavaScript", "Java", "C#", "AWS", "PowerBI", "Docker", "Angular"]

def pronadji_tehnologiju(tekst):
    nadjene = []
    tekst_lower = tekst.lower()
    for tech in tehnologije_lista:
        if tech.lower() in tekst_lower:
            nadjene.append(tech)
    return nadjene

df['tehnologije'] = df['naslov_i_opis'].apply(pronadji_tehnologiju)

def pronadji_lokaciju(tekst):
    tekst_upper = tekst.upper()
    if "REMOTE" in tekst_upper:
        return "Remote"
    elif "MOSTAR" in tekst_upper:
        return "Mostar"
    elif "SARAJEVO" in tekst_upper:
        return "Sarajevo"
    elif "BANJA LUKA" in tekst_upper:
        return "Banja Luka"
    else:
        return "BiH / Ostalo"


df['lokacija'] = df['naslov_i_opis'].apply(pronadji_lokaciju)

# Prikazujemo obrađene podatke
display(df[['naslov_i_opis', 'tehnologije', 'lokacija']].head(10))

,naslov_i_opis,tehnologije,lokacija
0,Poslovi,[],BiH / Ostalo
1,Objavi oglas,[],BiH / Ostalo
4,Remote BDR Admin,[],Remote
5,Frontend (React) Developer,[React],BiH / Ostalo
6,Senior Solution Engineer,[],BiH / Ostalo
8,Remote HR Administrator,[],Remote
9,Remote Implementation Consultant (Part Time),[],Remote
10,Remote Intermediate Python Developer,[Python],Remote
11,Remote Intermediate Ruby Developer,[],Remote
12,Remote Senior Front-End JavaScript Developer,"[JavaScript, Java]",Remote


In [25]:
import sqlite3

In [39]:
df['tehnologije'] = df['tehnologije'].apply(lambda x: ", ".join(x) if isinstance(x, list) else str(x)) # listu pretvaramo u string
conn = sqlite3.connect('it_poslovi.db')
cursor = conn.cursor()

df.to_sql('oglasi',conn,if_exists='replace', index=False)

print('Podatci su uspješno spremljeni u SQLite bazu!')


upit = "SELECT lokacija, COUNT(*) as broj_oglasa FROM oglasi GROUP BY lokacija"
df_iz_baze = pd.read_sql(upit, conn)

print("Brza SQL provjera (Broj oglasa po lokaciji)")
display(df_iz_baze)

conn.close()

Podatci su uspješno spremljeni u SQLite bazu!
Brza SQL provjera (Broj oglasa po lokaciji)


,lokacija,broj_oglasa
0,Banja Luka,1
1,BiH / Ostalo,12
2,Mostar,1
3,Remote,14
4,Sarajevo,1
